# Lab 7 - Diagnosing slow visuals (browser only)

You need nothing but this notebook. We use **semantic-link** (already installed
in Fabric) to trace each query and read the **storage engine (SE) vs formula
engine (FE)** split - the same information DAX Studio's Server Timings shows.

**The workflow:** run three slow queries, read the split, classify each as
SE-bound or FE-bound, then apply the cheapest fix and watch the split move.

## 1. Point at your model
This finds your own workspace for you. Run it and check the model name matches what `0-create-lab-models` built.

In [ ]:
# The notebook runs inside your own workspace, so it can resolve that itself.
# Only change MODEL if you want to measure a different one.
import sempy.fabric as fabric

BUILD     = "2026-08-24 15:58 66282618"          # stale copy check, see the build stamp printed below
WORKSPACE = fabric.resolve_workspace_name()
MODEL     = "07 Slow Visual Triage"

print(f"build     : {BUILD}")
print(f"workspace : {WORKSPACE}")
print(f"model     : {MODEL}")

## 2. The measurement helper
Run this once. `measure(dax, label)` traces a query and prints the SE/FE split.

In [ ]:
# Server Timings, in a notebook. semantic-link (sempy) is already installed in
# Fabric, so there is nothing to install for this cell.
import sempy.fabric as fabric
import pandas as pd
import time

# The events we want. QueryEnd = the whole query; VertiPaqSEQueryEnd = storage
# engine (SE) scans. Formula engine (FE) time is the remainder.
EVENTS = {
    "QueryEnd":                  ["EventClass", "EventSubclass", "TextData", "Duration", "CpuTime"],
    "VertiPaqSEQueryEnd":        ["EventClass", "EventSubclass", "TextData", "Duration", "CpuTime"],
    "VertiPaqSEQueryCacheMatch": ["EventClass", "TextData"],
}

def _col(df, name):
    """Find a column ignoring spaces/case (sempy names vary by version)."""
    key = name.replace(" ", "").lower()
    for c in df.columns:
        if c.replace(" ", "").lower() == key:
            return c
    return None

def _se_scans(logs, ec, sub):
    """SE scan events, minus the internal ones.

    Every VertiPaq scan raises TWO end events: the scan itself, and an internal
    event underneath it. DAX Studio hides the internal ones. Sum both and SE
    comes out at roughly double, which reports every query as SE-bound and
    leaves FE sitting at zero.
    """
    rows = logs[logs[ec] == "VertiPaqSEQueryEnd"]
    if sub is None or rows.empty:
        return rows
    s = (rows[sub].astype(str).str.strip().str.lower()
         .str.replace(r"\.0$", "", regex=True))
    outer = rows[~s.str.contains("internal") & (s != "10")]
    return outer if not outer.empty else rows      # unknown subclass, keep all

def measure(dax, label, settle=5, show_raw=False, model=None):
    """Run a DAX query, trace it, and return a total / SE / FE breakdown in ms.

    model= overrides the notebook default, so the same helper can measure a
    different model without editing the config cell.
    """
    model = model or MODEL
    with fabric.create_trace_connection(dataset=model, workspace=WORKSPACE) as tc:
        with tc.create_trace(EVENTS, "Perf trace") as tr:
            tr.start()
            fabric.evaluate_dax(dataset=model, dax_string=dax, workspace=WORKSPACE)
            time.sleep(settle)                 # let the trace events flush
            logs = tr.stop()

    ec  = _col(logs, "EventClass")
    dur = _col(logs, "Duration")
    sub = _col(logs, "EventSubclass")
    if show_raw or ec is None or dur is None:
        display(logs)                          # fall back to the raw trace
    if ec is None or dur is None:
        raise RuntimeError("trace has no EventClass/Duration column, see the frame above")

    ends = pd.to_numeric(logs.loc[logs[ec] == "QueryEnd", dur], errors="coerce").dropna()
    if ends.empty:
        display(logs)
        raise RuntimeError("no QueryEnd captured: raise settle and run the cell again")

    total = float(ends.max())
    scans = _se_scans(logs, ec, sub)
    se    = float(pd.to_numeric(scans[dur], errors="coerce").sum())
    cache = int((logs[ec] == "VertiPaqSEQueryCacheMatch").sum())
    fe    = max(total - se, 0.0)
    bound = "SE-bound" if total and se / total >= 0.5 else "FE-bound"

    # SE scans run in parallel, so their durations can add to more than the wall
    # clock. Say so rather than quietly reporting FE as zero.
    note = "   [SE scans overlap]" if se > total else ""
    if cache:
        note += f"   [{cache} cache hits, not a cold number]"
    print(f"{label:<22}  total {total:>7.0f} ms   SE {se:>7.0f} ms   "
          f"FE {fe:>7.0f} ms   {len(scans):>4} scans   -> {bound}{note}")
    return {"Query": label, "Total ms": total, "SE ms": se, "FE ms": fe,
            "Scans": len(scans), "Bound": bound, "Cache hits": cache}

## 3. Measure the three slow visuals

Each query below stands in for one slow visual. Run the cell, then read the
printed split. Do not tune anything yet - diagnose first.

In [ ]:
# Three deliberately slow queries. The two "Slow Sales" measures are planted in
# this model by 0-create-lab-models specifically for this lab.
q_a = """
EVALUATE
SUMMARIZECOLUMNS ( 'Date'[MonthYear], "Sales", [Slow Sales (SE)] )
"""   # Visual A - recomputes Quantity * UnitPrice * (1 - Discount) over every
      # row of a 3M-row fact (suspect: SE-bound)

q_b = """
EVALUATE
SUMMARIZECOLUMNS (
    'Product'[Category],
    "Heavy", [Slow Sales (FE)]
)
"""   # Visual B - context transition once per order number (suspect: FE-bound)

q_c = """
EVALUATE
ROW ( "Distinct orders", COUNTROWS ( VALUES ( 'Sales'[OrderNumber] ) ) )
"""   # Visual C - a ~1M-cardinality column (suspect: SE-bound). Returns a single
      # row on purpose: grouping by OrderNumber would ship ~1M rows back to
      # every notebook in the room at once.

results = [measure(q_a, "Visual A"), measure(q_b, "Visual B"), measure(q_c, "Visual C")]
summary = pd.DataFrame(results)
display(summary)

## 4. Read the split, then fix the cheapest thing

- **SE-bound** (SE is most of the time): you are scanning too much. Fix the
  model, add an aggregation, or cut cardinality.
- **FE-bound** (FE is most of the time): the measure logic is expensive.
  Simplify the DAX.

### Which fix goes with which visual

Try to reach these yourself from the split before you read the last column.

| | Diagnosis | Why | The fix |
|---|---|---|---|
| **A** | scanning too much | recomputes `Quantity * UnitPrice * (1 - Discount)` on all 3M rows, so it reads three columns and does arithmetic on every one | `SUM ( Sales[SalesAmount] )` - that value is already stored, so it reads one column and does no arithmetic |
| **B** | logic too expensive | a context transition once per order number, about a million of them, to compute something that does not depend on the order at all | `SUM ( Sales[SalesAmount] )` - summing per order and then adding those up is just summing |
| **C** | scanning too much | roughly 1M distinct values, and there is no cheap exact answer | no measure fixes this one. It needs the model to change - an aggregation (Module 4) or an order-grain table you can `COUNTROWS` (Module 6) |

Two honest notes, because someone will check:

- **A and B land on the same fix from different directions**, and that is worth
  saying rather than hiding. A is scanning columns it does not need; B is
  iterating when it does not need to. Different diagnosis, different reasoning,
  and this time the same destination. It will not always be.
- **A's fix is not bit-identical.** `SalesAmount` is stored rounded to two
  decimals per row; the slow measure rounds nothing until the end. Across 3M
  rows those differ in the last significant figures. The right answer is to
  decide whether you care, not to pretend they match.
- **C has no free fix**, and that is the most useful lesson of the three. On
  another storage mode you could reach for `APPROXIMATEDISTINCTCOUNT`, which
  buys speed with exactness - fine for a headline tile nobody reconciles, not
  fine for anything a finance team signs. **It is unsupported in Direct Lake, so
  do not try it on your model.** That leaves the only real answer: change the
  model. **You have already been shown both versions of that today** - the
  user-defined aggregation with `alternateOf` in Module 4, and the order-grain
  table you can `COUNTROWS` in Module 6. Neither is a measure edit, which is
  exactly why C stays unfixed in this lab. Some problems you diagnose here and
  solve somewhere else.

### Now fix one, step by step

Fix **Visual A**. Its measure recomputes a value the model already stores, so
the fix is to stop recomputing it.

1. Open your workspace and click the **07 Slow Visual Triage** semantic model.
2. Click **Open data model** at the top.
3. In the **Data** pane expand **Sales** and click the measure
   **`Slow Sales (SE)`**.
4. In the formula bar, replace the whole expression with:

   `SUM ( Sales[SalesAmount] )`

5. Commit it (the tick, or Enter), then come back here and run the cell below.

Visual A should flip from **SE-bound** to a query so cheap the split stops
being interesting - which is the point. You removed three column reads and a
row-by-row multiplication by using a column somebody already computed upstream.

> **The number will move slightly, and that is real.** `SalesAmount` is stored
> rounded to 2dp per row; the slow measure rounds nothing until the end. On
> this model that is **575,871,342.02** against **575,871,284.54**. Recomputing
> a stored value is exactly how a "harmless" rewrite quietly changes a number.

> ⚠️ **Do not edit `Slow Sales (FE)`.** Lab 8 uses it as the slow half of its
> benchmark. Change it here and Lab 8 will compare two identical measures and
> show you nothing.

**If you see `[n cache hits]`** the engine answered from cache rather than
doing the work, so that timing is not comparable to the first run. Editing the
model invalidates the cache, so a genuine fix will not show it. Re-running the
same query against an unchanged model will.

**If you see `[SE scans overlap]`** the storage engine ran several scans at
once and their durations add to more than the elapsed time. The FE number is
a floor in that case, not an exact figure.

In [ ]:
# After you fix ONE visual in the model, re-measure just that query:
display(pd.DataFrame([measure(q_a, "Visual A (after fix)")]))

## Optional - if you happen to have DAX Studio

The same three queries are in `lab07-slow-visuals.dax`. Paste them into DAX
Studio with **Server Timings** on to see the identical SE/FE breakdown. This is
completely optional; the notebook above already gives you everything you need.